In [ ]:
import nltools
from nltools.data import Brain_Data
from nltools.mask import expand_mask, roi_to_brain
from nilearn import maskers
import nibabel as nib

import os
import glob

import numpy as np
import pandas as pd
import statistics
import itertools
import random
import time

import scipy.stats
from scipy.stats import ttest_ind_from_stats, mannwhitneyu, pearsonr
import statsmodels.api as sm
from statsmodels.formula.api import ols
from nltools.stats import isc, fdr, threshold

import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.plotting import view_img_on_surf, view_img, plot_surf_roi, plot_glass_brain, plot_stat_map

%matplotlib inline
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
mask = Brain_Data('/path/to/mask/shen_2mm_268_parcellation.nii.gz')
mask_x = expand_mask(mask)

mask.plot()

# Face TR analysis

## 1. MovieDM ISC in each group

In [ ]:
data_dir_ASD = '/path/to/data/directory'

sub_list_ASD = [os.path.basename(x).split('_')[0] for x in glob.glob(os.path.join(data_dir_ASD, '*movieDM*n268*csv'))]
sub_list_ASD.sort()

In [ ]:
# Saving time-series data for each participant

sub_timeseries_ASD = {}

for sub in sub_list_ASD:
    sub_data_ASD = pd.read_csv(os.path.join(data_dir_ASD, f'{sub}_movieDM_Average_ROI_n268.csv'))
    sub_data_ASD.reset_index(inplace=True, drop=True)
    sub_timeseries_ASD[sub] = sub_data_ASD

In [ ]:
data_dir_control = '/path/to/data/directory'

sub_list_control = [os.path.basename(x).split('_')[0] for x in glob.glob(os.path.join(data_dir_control, '*movieDM*n268*csv'))]
sub_list_control.sort()

In [ ]:
# Saving time-series data for each participant

sub_timeseries_control = {}

for sub in sub_list_control:
    sub_data_control = pd.read_csv(os.path.join(data_dir_control, f'{sub}_movieDM_Average_ROI_n268.csv'))
    sub_data_control.reset_index(inplace=True, drop=True)
    sub_timeseries_control[sub] = sub_data_control

In [ ]:
#Face TR selection in MovieDM

sub_timeseries_ASD = {}

for sub in sub_list_ASD:
    sub_data_ASD = pd.read_csv(os.path.join(data_dir_ASD, f'{sub}_movieDM_Average_ROI_n268.csv'))
    sub_data_ASD.reset_index(inplace=True, drop=True)

    df1 = sub_data_ASD.iloc[22:36+1]
    df2 = sub_data_ASD.iloc[41:43+1]
    df3 = sub_data_ASD.iloc[46:46+1]
    df4 = sub_data_ASD.iloc[53:56+1]
    df5 = sub_data_ASD.iloc[60:64+1]
    df6 = sub_data_ASD.iloc[67:76+1]
    df7 = sub_data_ASD.iloc[78:82+1]
    df8 = sub_data_ASD.iloc[109:114+1]
    df9 = sub_data_ASD.iloc[120:130+1]
    df10 = sub_data_ASD.iloc[147:151+1]
    df11 = sub_data_ASD.iloc[184:206+1]
    df12 = sub_data_ASD.iloc[214:216+1]
    df13 = sub_data_ASD.iloc[232:238+1]
    df14 = sub_data_ASD.iloc[258:269+1]
    df15 = sub_data_ASD.iloc[291:305+1]
    df16 = sub_data_ASD.iloc[316:320+1]
    df17 = sub_data_ASD.iloc[332:337+1]
    df18 = sub_data_ASD.iloc[393:395+1]
    df19 = sub_data_ASD.iloc[404:410+1]
    df20 = sub_data_ASD.iloc[414:422+1]
    df21 = sub_data_ASD.iloc[426:428+1]
    df22 = sub_data_ASD.iloc[438:442+1]
    df23 = sub_data_ASD.iloc[489:497+1]
    df24 = sub_data_ASD.iloc[509:513+1]
    df25 = sub_data_ASD.iloc[517:519+1]
    df26 = sub_data_ASD.iloc[523:524+1]
    df27 = sub_data_ASD.iloc[552:554+1]
    df28 = sub_data_ASD.iloc[562:563+1]
    df29 = sub_data_ASD.iloc[578:582+1]
    df30 = sub_data_ASD.iloc[591:594+1]
    df31 = sub_data_ASD.iloc[639:641+1]
    df32 = sub_data_ASD.iloc[659:663+1]
    df33 = sub_data_ASD.iloc[690:694+1]
    df34 = sub_data_ASD.iloc[743:745+1]
        

    df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13, df14, df15, df16, df17, df18, df19, df20, df21, df22, 
                   df23, df24, df25, df26, df27, df28, df29, df30, df31, df32, df33, df34])

    sub_data_ASD = df
    sub_timeseries_ASD[sub] = sub_data_ASD
print ('dataframe for ASD is ready...(',len(sub_data_ASD),"TRs)")



sub_timeseries_control = {}

for sub in sub_list_control:
    sub_data_control = pd.read_csv(os.path.join(data_dir_control, f'{sub}_movieDM_Average_ROI_n268.csv'))
    sub_data_control.reset_index(inplace=True, drop=True)

    df1 = sub_data_control.iloc[22:36+1]
    df2 = sub_data_control.iloc[41:43+1]
    df3 = sub_data_control.iloc[46:46+1]
    df4 = sub_data_control.iloc[53:56+1]
    df5 = sub_data_control.iloc[60:64+1]
    df6 = sub_data_control.iloc[67:76+1]
    df7 = sub_data_control.iloc[78:82+1]
    df8 = sub_data_control.iloc[109:114+1]
    df9 = sub_data_control.iloc[120:130+1]
    df10 = sub_data_control.iloc[147:151+1]
    df11 = sub_data_control.iloc[184:206+1]
    df12 = sub_data_control.iloc[214:216+1]
    df13 = sub_data_control.iloc[232:238+1]
    df14 = sub_data_control.iloc[258:269+1]
    df15 = sub_data_control.iloc[291:305+1]
    df16 = sub_data_control.iloc[316:320+1]
    df17 = sub_data_control.iloc[332:337+1]
    df18 = sub_data_control.iloc[393:395+1]
    df19 = sub_data_control.iloc[404:410+1]
    df20 = sub_data_control.iloc[414:422+1]
    df21 = sub_data_control.iloc[426:428+1]
    df22 = sub_data_control.iloc[438:442+1]
    df23 = sub_data_control.iloc[489:497+1]
    df24 = sub_data_control.iloc[509:513+1]
    df25 = sub_data_control.iloc[517:519+1]
    df26 = sub_data_control.iloc[523:524+1]
    df27 = sub_data_control.iloc[552:554+1]
    df28 = sub_data_control.iloc[562:563+1]
    df29 = sub_data_control.iloc[578:582+1]
    df30 = sub_data_control.iloc[591:594+1]
    df31 = sub_data_control.iloc[639:641+1]
    df32 = sub_data_control.iloc[659:663+1]
    df33 = sub_data_control.iloc[690:694+1]
    df34 = sub_data_control.iloc[743:745+1]
        

    df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13, df14, df15, df16, df17, df18, df19, df20, df21, df22, 
                   df23, df24, df25, df26, df27, df28, df29, df30, df31, df32, df33, df34])

    sub_data_control = df
    sub_timeseries_control[sub] = sub_data_control
print ('dataframe for control is ready...(',len(sub_data_control),"TRs)")

In [ ]:
roi = 67 # example ROI

def get_subject_roi_ASD(sub_data_ASD, roi):
    sub_rois_ASD = {}
    for sub in sub_data_ASD:
        sub_rois_ASD[sub] = sub_data_ASD[sub].iloc[:, roi]
    return pd.DataFrame(sub_rois_ASD)

sub_rois_ASD = get_subject_roi_ASD(sub_timeseries_ASD, roi)
sub_rois_ASD

In [ ]:
roi = 67 # example ROI

def get_subject_roi_control(sub_data_control, roi):
    sub_rois_control = {}
    for sub in sub_data_control:
        sub_rois_control[sub] = sub_data_control[sub].iloc[:, roi]
    return pd.DataFrame(sub_rois_control)

sub_rois_control = get_subject_roi_control(sub_timeseries_control, roi)
sub_rois_control

In [ ]:
# Compute ISC during selected face scenes for each ROI

isc_r_ASD, isc_p_ASD = {}, {}

for roi in range (268):
    stats = isc(get_subject_roi_ASD(sub_timeseries_ASD, roi), n_samples=30000, metric='median', method='bootstrap')
    isc_r_ASD[roi], isc_p_ASD[roi] = stats['isc'], stats['p']

In [ ]:
# Save the ISC results

data = {
    'ROI': list(isc_r_ASD.keys()),
    'ISC_r': list(isc_r_ASD.values()),
    'ISC_p': list(isc_p_ASD.values())
}
df = pd.DataFrame(data)

# Save results to csv
df.to_csv('/output/path/ASD_isc_DM_face.csv', index=False)

In [ ]:
# Compute ISC during selected face scenes for each ROI(region of interest)

isc_r_control, isc_p_control = {}, {}

for roi in range (268):
    stats = isc(get_subject_roi_control(sub_timeseries_control, roi), n_samples=30000, metric='median', method='bootstrap')
    isc_r_control[roi], isc_p_control[roi] = stats['isc'], stats['p']

In [ ]:
# Save the ISC results

data = {
    'ROI': list(isc_r_control.keys()),
    'ISC_r': list(isc_r_control.values()),
    'ISC_p': list(isc_p_control.values())
}
df = pd.DataFrame(data)

# Save results to csv
df.to_csv('/output/path/control_isc_DM_face.csv', index=False)

## 2. MovieDM group comparison

In [ ]:
# Compute pairwise correlations for each ROI in the ASD group

col_name = []
for i in range(268):
    col_name.append("ASD_roi_" + str(i))

pair_list_ASD = [pd.DataFrame()]

for roi in range(268):

    sub_rois_ASD = get_subject_roi_ASD(sub_timeseries_ASD, roi)
    
    correlations = {}
    columns = sub_rois_ASD.columns.tolist()

    for col_a, col_b in itertools.combinations(columns, 2):
        correlations[col_a + '__' + col_b] = pearsonr(sub_rois_ASD.loc[:, col_a], sub_rois_ASD.loc[:, col_b])

    result = pd.DataFrame.from_dict(correlations, orient='index')
    result.columns = ['PCC', 'p-value']
    corr_result = pd.DataFrame.from_dict(correlations, orient='index')

    corr_list = corr_result.iloc[:,0].values.tolist()
    corr_list = pd.DataFrame.from_dict(corr_list)
    pair_list_ASD.append(corr_list)


pair_list_ASD = pd.concat(pair_list_ASD, axis = 1)
pair_list_ASD.columns = col_name
pair_list_ASD.fillna(0, inplace=True)


# Compute pairwise correlations for each ROI in the TD group

col_name = []
for i in range(268):
    col_name.append("control_roi_" + str(i))

pair_list_control = [pd.DataFrame()]

for roi in range(268):

    sub_rois_control = get_subject_roi_control(sub_timeseries_control, roi)

    correlations = {}
    columns = sub_rois_control.columns.tolist()

    for col_a, col_b in itertools.combinations(columns, 2):
        correlations[col_a + '__' + col_b] = pearsonr(sub_rois_control.loc[:, col_a], sub_rois_control.loc[:, col_b])

    result = pd.DataFrame.from_dict(correlations, orient='index')
    result.columns = ['PCC', 'p-value']
    corr_result = pd.DataFrame.from_dict(correlations, orient='index')

    corr_list = corr_result.iloc[:,0].values.tolist()
    corr_list = pd.DataFrame.from_dict(corr_list)
    pair_list_control.append(corr_list)


pair_list_control = pd.concat(pair_list_control, axis = 1)
pair_list_control.columns = col_name
pair_list_control.fillna(0, inplace=True)

In [ ]:
# Compute the difference in pairwise correlation between ASD and TD for each ROI

results = []

for roi in range (268):
        # Fisher r-to-z transformation
        ASD = np.arctanh(pair_list_ASD.iloc[:,roi])
        control = np.arctanh(pair_list_control.iloc[:,roi])

        u_test = scipy.stats.mannwhitneyu(ASD,control)

        # Compute the median pairwise correlation in each group (ISC)
        median_ASD = pair_list_ASD.iloc[:,roi].median()
        median_control = pair_list_control.iloc[:,roi].median()

        results.append({
        'ROI': roi,
        'u_statistic': u_test.statistic,
        'p_value': u_test.pvalue,
        'median_ASD': median_ASD,
        'median_control': median_control
    })

results_df = pd.DataFrame(results)

# Save to CSV
results_df.to_csv('/output/path/utest_face_movieDM.csv', index=False)

## 3. MovieTP ISC in each group

In [ ]:
data_dir_ASD = '/path/to/data/directory'

sub_list_ASD = [os.path.basename(x).split('_')[0] for x in glob.glob(os.path.join(data_dir_ASD, '*movieTP*n268*csv'))]
sub_list_ASD.sort()

In [ ]:
# Saving time-series data for each participant

sub_timeseries_ASD = {}

for sub in sub_list_ASD:
    sub_data_ASD = pd.read_csv(os.path.join(data_dir_ASD, f'{sub}_movieTP_Average_ROI_n268.csv'))
    sub_data_ASD.reset_index(inplace=True, drop=True)
    sub_timeseries_ASD[sub] = sub_data_ASD

In [ ]:
data_dir_control = '/path/to/data/directory'

sub_list_control = [os.path.basename(x).split('_')[0] for x in glob.glob(os.path.join(data_dir_control, '*movieTP*n268*csv'))]
sub_list_control.sort()

In [ ]:
# Saving time-series data for each participant

sub_timeseries_control = {}

for sub in sub_list_control:
    sub_data_control = pd.read_csv(os.path.join(data_dir_control, f'{sub}_movieTP_Average_ROI_n268.csv'))
    sub_data_control.reset_index(inplace=True, drop=True)
    sub_timeseries_control[sub] = sub_data_control

In [ ]:
#Face TR selection in MovieTP

sub_timeseries_ASD = {}

for sub in sub_list_ASD:
    sub_data_ASD = pd.read_csv(os.path.join(data_dir_ASD, f'{sub}_movieTP_Average_ROI_n268.csv'))
    sub_data_ASD.reset_index(inplace=True, drop=True)

    df1=sub_data_ASD.iloc[11:15+1]
    df2=sub_data_ASD.iloc[21:28+1]
    df3=sub_data_ASD.iloc[32:36+1]
    df4=sub_data_ASD.iloc[37:45+1]
    df5=sub_data_ASD.iloc[47:53+1]
    df6=sub_data_ASD.iloc[55:60+1]
    df7=sub_data_ASD.iloc[62:65+1]
    df8=sub_data_ASD.iloc[76:76+1]
    df9=sub_data_ASD.iloc[83:84+1]
    df10=sub_data_ASD.iloc[87:88+1]
    df11=sub_data_ASD.iloc[97:98+1]
    df12=sub_data_ASD.iloc[100:103+1]
    df13=sub_data_ASD.iloc[122:127+1]
    df14=sub_data_ASD.iloc[131:132+1]
    df15=sub_data_ASD.iloc[139:143+1]
    df16=sub_data_ASD.iloc[146:147+1]
    df17=sub_data_ASD.iloc[150:153+1]
    df18=sub_data_ASD.iloc[165:171+1]
    df19=sub_data_ASD.iloc[176:178+1]
    df20=sub_data_ASD.iloc[194:197+1]
    df21=sub_data_ASD.iloc[201:204+1]
    df22=sub_data_ASD.iloc[230:234+1]
        

    df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13, df14, df15, df16, df17, df18, df19, df20, df21, df22])

    sub_data_ASD = df
    sub_timeseries_ASD[sub] = sub_data_ASD
print ('dataframe for ASD is ready...(',len(sub_data_ASD),"TRs)")



sub_timeseries_control = {}

for sub in sub_list_control:
    sub_data_control = pd.read_csv(os.path.join(data_dir_control, f'{sub}_movieTP_Average_ROI_n268.csv'))
    sub_data_control.reset_index(inplace=True, drop=True)

    df1 = sub_data_control.iloc[11:15+1]
    df2 = sub_data_control.iloc[21:28+1]
    df3 = sub_data_control.iloc[32:36+1]
    df4 = sub_data_control.iloc[37:45+1]
    df5 = sub_data_control.iloc[47:53+1]
    df6 = sub_data_control.iloc[55:60+1]
    df7 = sub_data_control.iloc[62:65+1]
    df8 = sub_data_control.iloc[76:76+1]
    df9 = sub_data_control.iloc[83:84+1]
    df10 = sub_data_control.iloc[87:88+1]
    df11 = sub_data_control.iloc[97:98+1]
    df12 = sub_data_control.iloc[100:103+1]
    df13 = sub_data_control.iloc[122:127+1]
    df14 = sub_data_control.iloc[131:132+1]
    df15 = sub_data_control.iloc[139:143+1]
    df16 = sub_data_control.iloc[146:147+1]
    df17 = sub_data_control.iloc[150:153+1]
    df18 = sub_data_control.iloc[165:171+1]
    df19 = sub_data_control.iloc[176:178+1]
    df20 = sub_data_control.iloc[194:197+1]
    df21 = sub_data_control.iloc[201:204+1]
    df22 = sub_data_control.iloc[230:234+1]

        

    df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13, df14, df15, df16, df17, df18, df19, df20, df21, df22])

    sub_data_control = df
    sub_timeseries_control[sub] = sub_data_control
print ('dataframe for control is ready...(',len(sub_data_control),"TRs)")

In [ ]:
roi = 67 # example ROI

def get_subject_roi_ASD(sub_data_ASD, roi):
    sub_rois_ASD = {}
    for sub in sub_data_ASD:
        sub_rois_ASD[sub] = sub_data_ASD[sub].iloc[:, roi]
    return pd.DataFrame(sub_rois_ASD)

sub_rois_ASD = get_subject_roi_ASD(sub_timeseries_ASD, roi)
sub_rois_ASD

In [ ]:
roi = 67 # example ROI

def get_subject_roi_control(sub_data_control, roi):
    sub_rois_control = {}
    for sub in sub_data_control:
        sub_rois_control[sub] = sub_data_control[sub].iloc[:, roi]
    return pd.DataFrame(sub_rois_control)

sub_rois_control = get_subject_roi_control(sub_timeseries_control, roi)
sub_rois_control

In [ ]:
# Compute ISC during selected face scenes for each ROI

isc_r_ASD, isc_p_ASD = {}, {}

for roi in range (268):
    stats = isc(get_subject_roi_ASD(sub_timeseries_ASD, roi), n_samples=30000, metric='median', method='bootstrap')
    isc_r_ASD[roi], isc_p_ASD[roi] = stats['isc'], stats['p']

In [ ]:
# Save the ISC results

data = {
    'ROI': list(isc_r_ASD.keys()),
    'ISC_r': list(isc_r_ASD.values()),
    'ISC_p': list(isc_p_ASD.values())
}
df = pd.DataFrame(data)

# Save results ro csv
df.to_csv('output/path/ASD_isc_TP_face.csv', index=False)

In [ ]:
# Compute ISC during selected face scenes for each ROI

isc_r_control, isc_p_control = {}, {}

for roi in range (268):
    stats = isc(get_subject_roi_control(sub_timeseries_control, roi), n_samples=30000, metric='median', method='bootstrap')
    isc_r_control[roi], isc_p_control[roi] = stats['isc'], stats['p']

In [ ]:
# Save the ISC results

data = {
    'ROI': list(isc_r_control.keys()),
    'ISC_r': list(isc_r_control.values()),
    'ISC_p': list(isc_p_control.values())
}
df = pd.DataFrame(data)

# Save results ro csv
df.to_csv('output/path/control_isc_TP_face.csv', index=False)

## 4. MovieTP group comparison

In [ ]:
# Compute pairwise correlations for each ROI in the ASD group

col_name = []
for i in range(268):
    col_name.append("ASD_roi_" + str(i))

pair_list_ASD = [pd.DataFrame()]

for roi in range(268):

    sub_rois_ASD = get_subject_roi_ASD(sub_timeseries_ASD, roi)
    
    correlations = {}
    columns = sub_rois_ASD.columns.tolist()

    for col_a, col_b in itertools.combinations(columns, 2):
        correlations[col_a + '__' + col_b] = pearsonr(sub_rois_ASD.loc[:, col_a], sub_rois_ASD.loc[:, col_b])

    result = pd.DataFrame.from_dict(correlations, orient='index')
    result.columns = ['PCC', 'p-value']
    corr_result = pd.DataFrame.from_dict(correlations, orient='index')

    corr_list = corr_result.iloc[:,0].values.tolist()
    corr_list = pd.DataFrame.from_dict(corr_list)
    pair_list_ASD.append(corr_list)


pair_list_ASD = pd.concat(pair_list_ASD, axis = 1)
pair_list_ASD.columns = col_name
pair_list_ASD.fillna(0, inplace=True)


# Compute pairwise correlations for each ROI in the TD group

col_name = []
for i in range(268):
    col_name.append("control_roi_" + str(i))

pair_list_control = [pd.DataFrame()]

for roi in range(268):

    sub_rois_control = get_subject_roi_control(sub_timeseries_control, roi)

    correlations = {}
    columns = sub_rois_control.columns.tolist()

    for col_a, col_b in itertools.combinations(columns, 2):
        correlations[col_a + '__' + col_b] = pearsonr(sub_rois_control.loc[:, col_a], sub_rois_control.loc[:, col_b])

    result = pd.DataFrame.from_dict(correlations, orient='index')
    result.columns = ['PCC', 'p-value']
    corr_result = pd.DataFrame.from_dict(correlations, orient='index')

    corr_list = corr_result.iloc[:,0].values.tolist()
    corr_list = pd.DataFrame.from_dict(corr_list)
    pair_list_control.append(corr_list)


pair_list_control = pd.concat(pair_list_control, axis = 1)
pair_list_control.columns = col_name
pair_list_control.fillna(0, inplace=True)

In [ ]:
# Compute the difference in pairwise correlation between ASD and TD for each ROI

results = []

for roi in range (268):
        # Fisher r-to-z transformation
        ASD = np.arctanh(pair_list_ASD.iloc[:,roi])
        control = np.arctanh(pair_list_control.iloc[:,roi])

        u_test = scipy.stats.mannwhitneyu(ASD,control)

        # Compute the median pairwise correlation in each group (ISC)
        median_ASD = pair_list_ASD.iloc[:,roi].median()
        median_control = pair_list_control.iloc[:,roi].median()

        results.append({
        'ROI': roi,
        'u_statistic': u_test.statistic,
        'p_value': u_test.pvalue,
        'median_ASD': median_ASD,
        'median_control': median_control
    })

results_df = pd.DataFrame(results)

# Save to CSV
results_df.to_csv('output/path/utest_face_movieTP.csv', index=False)